# Michaelis–Menten Kinetics and Lineweaver–Burk Plots

### Enzyme kinetics, parameter estimation, and reversible inhibition

This notebook has three parts:

1. **Inspect enzyme kinetics data**
2. **Use a Lineweaver–Burk plot to estimate** $K_M$ **and** $V_{\max}$
3. **Use Lineweaver–Burk plots to recognize reversible inhibition patterns**

> **Part 3 is for observation and interpretation only.** You do not need to derive the inhibition equations or perform additional fitting.

---


## Part 1: Inspect the enzyme kinetics data

We will start with a small enzyme kinetics dataset and plot the initial velocity, $V_0$, as a function of substrate concentration, $[S]$.

As $[S]$ increases, ask:

- Does $V_0$ continue to increase linearly?
- Does the reaction begin to approach a maximum velocity?

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# Substrate concentrations [S] (mM) and initial velocities V0 (μM/min)
S = np.array([0.5, 1, 2, 4, 8, 16], dtype=float)
V = np.array([0.05, 0.09, 0.16, 0.26, 0.35, 0.42], dtype=float)

plt.figure(figsize=(6,4))
plt.scatter(S, V)
plt.xlabel('[S] (mM)')
plt.ylabel('Initial velocity, V0 (μM/min)')
plt.title('Enzyme kinetics data')
plt.show()


## Part 2: Michaelis–Menten kinetics using a Lineweaver–Burk plot

For a simple Michaelis–Menten enzyme,

$$
V_0 = \frac{V_{\max}[S]}{K_M + [S]}
$$

Taking the reciprocal gives the **Lineweaver–Burk equation**:

$$
\frac{1}{V_0}
=
\frac{K_M}{V_{\max}}\frac{1}{[S]}
+
\frac{1}{V_{\max}}
$$

This has the same form as a straight line,

$$
y = mx + b
$$

with

$$
\text{slope} = \frac{K_M}{V_{\max}}
$$

and

$$
y\text{-intercept} = \frac{1}{V_{\max}}.
$$

Therefore,

$$
V_{\max} = \frac{1}{y\text{-intercept}}
$$

and

$$
K_M = (\text{slope})V_{\max}.
$$

We can use the transformed data to estimate $K_M$ and $V_{\max}$, and then place those values back into the Michaelis–Menten equation to compare the fitted curve with the original data.

---


In [ ]:
# Transform the data
inv_S = 1 / S
inv_V = 1 / V

# Linear regression
slope, intercept, r_value, p_value, std_err = linregress(inv_S, inv_V)

# Recover Michaelis–Menten parameters
Vmax_LB = 1 / intercept
Km_LB = slope * Vmax_LB

print(f'Vmax = {Vmax_LB:.3f} μM/min')
print(f'Km   = {Km_LB:.3f} mM')

# Lineweaver–Burk plot
xfit = np.linspace(0, inv_S.max() * 1.05, 200)

plt.figure(figsize=(6,4))
plt.scatter(inv_S, inv_V, label='data')
plt.plot(xfit, slope*xfit + intercept, label='linear fit')
plt.xlabel('1/[S] (1/mM)')
plt.ylabel('1/V0 (min/μM)')
plt.title('Lineweaver–Burk plot')
plt.legend()
plt.show()


In [ ]:
# Put the fitted parameters back into the Michaelis–Menten equation
S_fit = np.linspace(0, S.max(), 300)
V_fit = Vmax_LB * S_fit / (Km_LB + S_fit)

plt.figure(figsize=(6,4))
plt.scatter(S, V, label='data')
plt.plot(S_fit, V_fit, label='Michaelis–Menten curve from LB fit')
plt.xlabel('[S] (mM)')
plt.ylabel('Initial velocity, V0 (μM/min)')
plt.title('Michaelis–Menten kinetics')
plt.legend()
plt.show()


## Part 3: Reversible inhibition and Lineweaver–Burk patterns

Now keep the same enzyme and ask what happens when a reversible inhibitor is added.

The code below generates Lineweaver–Burk plots at several inhibitor concentrations for three mechanisms:

- **Competitive inhibition**
- **Uncompetitive inhibition**
- **Pure noncompetitive inhibition**

> **You do not need to derive these equations or fit the data.** Focus on the pattern of the lines and connect it to what we discussed in class.

For each plot, ask:

- Do the lines intersect? If so, **where**?
- Are the lines **parallel**?
- What does the pattern imply about the apparent $K_M$ and $V_{\max}$?

---


In [ ]:
# Use the Michaelis–Menten parameters estimated above
Vmax = Vmax_LB
Km = Km_LB

# Example inhibitor concentrations and inhibition constant
I_values = [0, 1, 3]
Ki = 1.0

S_inhib = np.array([0.5, 1, 2, 4, 8, 16], dtype=float)

def competitive_velocity(S, I):
    alpha = 1 + I/Ki
    return Vmax * S / (alpha*Km + S)

def uncompetitive_velocity(S, I):
    alpha_prime = 1 + I/Ki
    return Vmax * S / (Km + alpha_prime*S)

def noncompetitive_velocity(S, I):
    alpha = 1 + I/Ki
    return (Vmax/alpha) * S / (Km + S)

def plot_lb_for_inhibition(velocity_function, title):
    plt.figure(figsize=(6,4))
    for I in I_values:
        v = velocity_function(S_inhib, I)
        x = 1 / S_inhib
        y = 1 / v

        slope_i, intercept_i, *_ = linregress(x, y)
        xline = np.linspace(0, x.max()*1.05, 200)

        plt.plot(xline, slope_i*xline + intercept_i, label=f'[I] = {I}')
        plt.scatter(x, y)

    plt.xlabel('1/[S] (1/mM)')
    plt.ylabel('1/V0 (min/μM)')
    plt.title(title)
    plt.legend()
    plt.show()


In [ ]:
plot_lb_for_inhibition(
    competitive_velocity,
    'Lineweaver–Burk: competitive inhibition'
)


In [ ]:
plot_lb_for_inhibition(
    uncompetitive_velocity,
    'Lineweaver–Burk: uncompetitive inhibition'
)


In [ ]:
plot_lb_for_inhibition(
    noncompetitive_velocity,
    'Lineweaver–Burk: pure noncompetitive inhibition'
)


## Take-home message

Lineweaver–Burk plots provide a visual way to distinguish reversible inhibition mechanisms because each mechanism changes the apparent $K_M$ and/or $V_{\max}$ in a characteristic way.

| Inhibition mechanism | Apparent $K_M$ | Apparent $V_{\max}$ | Lineweaver–Burk pattern |
|---|---:|---:|---|
| **Competitive** | Increases | Unchanged | Lines intersect at the **y-axis** |
| **Uncompetitive** | Decreases | Decreases | Lines are **parallel** |
| **Pure noncompetitive** | Unchanged | Decreases | Lines intersect at the **x-axis** |

> **No additional calculations are required for Part 3.**
